In [7]:
#INSTALAR PYSPARK

%pip install pyspark
print("Pyspark instalado")

Pyspark instalado


In [8]:
#IMPORTAR LIBRERIAS
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pathlib import Path
import pandas as pd
import time

print("Librerias importadas---------------")

Librerias importadas---------------


In [9]:
#CREAR SESION DE SPARK

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HRTech_PeopleAnalytics_Arquitectura") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Paralelismo: {spark.sparkContext.defaultParallelism}")

Spark version: 4.0.5
Paralelismo: 2


In [10]:
#DEFINIR RUTAS

BASE_DIR = Path("/content")
DATA_PROCESSED = BASE_DIR / "datos" / "procesados"
DATA_ANALYTICS = BASE_DIR / "datos" / "analiticos"
DOCUMENTOS = BASE_DIR / "documentos"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_ANALYTICS.mkdir(parents=True, exist_ok=True)
DOCUMENTOS.mkdir(parents=True, exist_ok=True)

print(f"Datos procesados: {DATA_PROCESSED}")
print(f"Datos analiticos: {DATA_ANALYTICS}")
print(f"Documentos: {DOCUMENTOS}")

Datos procesados: /content/datos/procesados
Datos analiticos: /content/datos/analiticos
Documentos: /content/documentos


In [11]:
#CARGAR DATASET LIMPIO
ruta_entrada = DATA_PROCESSED / "dataset_limpio_HRTech_People_Analytics.csv"

print(f"Leyendo: {ruta_entrada}")

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(ruta_entrada))

print("Dataset cargado en Spark")
print(f"Filas: {df.count()}")
print(f"Columnas: {len(df.columns)}")

Leyendo: /content/datos/procesados/dataset_limpio_HRTech_People_Analytics.csv
Dataset cargado en Spark
Filas: 10
Columnas: 22


In [12]:
#1. EXPLORACION BASICA - ESQUEMA Y ESTRUCTURA DEL DATASET
print("Esquema de datos:-------------")
df.printSchema()

print("\nPrimeras filas: ")
df.show(10, truncate=False)

Esquema de datos:-------------
root
 |-- id_empleado: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- desempeño: integer (nullable = true)
 |-- salario: double (nullable = true)
 |-- tiempo_empresa_anios: integer (nullable = true)
 |-- estado_empleado: string (nullable = true)
 |-- salario_anual: double (nullable = true)
 |-- id_capacitacion: integer (nullable = true)
 |-- curso: string (nullable = true)
 |-- horas: integer (nullable = true)
 |-- resultado: string (nullable = true)
 |-- horas_capacitacion: integer (nullable = true)
 |-- satisfaccion: integer (nullable = true)
 |-- horas_extra: integer (nullable = true)
 |-- salario_por_hora: double (nullable = true)
 |-- anios_redondeados: integer (nullable = true)
 |-- es_antiguo: boolean (nullable = true)
 |-- categoria_antiguedad: string (nullable = true)
 |-- rango_edad: string (nullable = true)
 |-- requiere_capacitacion: boolean 

In [13]:
#ESTADISTICAS DESCRIPTIVAS
print("Estadisticas descriptivas (describe):-------------")
df.describe().show(truncate=False)

#ESTADISTICAS SOLO DE COLUMNAS NUMERICAS CLAVE
print("\nEstadisticas de columnas clave:")
df.describe("edad", "desempeño", "salario", "satisfaccion", "tiempo_empresa_anios").show(truncate=False)

Estadisticas descriptivas (describe):-------------
+-------+------------------+------+------------+-----------------+-----------------+------------------+--------------------+---------------+------------------+------------------+-----------------+------------------+-----------+------------------+------------------+-----------------+------------------+-----------------+--------------------+--------------+
|summary|id_empleado       |nombre|departamento|edad             |desempeño        |salario           |tiempo_empresa_anios|estado_empleado|salario_anual     |id_capacitacion   |curso            |horas             |resultado  |horas_capacitacion|satisfaccion      |horas_extra      |salario_por_hora  |anios_redondeados|categoria_antiguedad|rango_edad    |
+-------+------------------+------+------------+-----------------+-----------------+------------------+--------------------+---------------+------------------+------------------+-----------------+------------------+-----------+--------

In [14]:
#PARTICIONES - CON SPARK
print("Particiones iniciales:", df.rdd.getNumPartitions())

Particiones iniciales: 1


In [15]:
#TRANSFORMACION 1 - FILTER: EMPLEADOS ACTIVOS
df_activos = df.filter(F.col("estado_empleado") == "Activo")

print(f"Registros totales: {df.count()}")
print(f"Empleados activos: {df_activos.count()}")
print(f"Empleados inactivos: {df.count() - df_activos.count()}")

df_activos.select("id_empleado", "nombre", "departamento", "estado_empleado").show(truncate=False)

Registros totales: 10
Empleados activos: 7
Empleados inactivos: 3
+-----------+------+------------+---------------+
|id_empleado|nombre|departamento|estado_empleado|
+-----------+------+------------+---------------+
|1          |Ana   |Ventas      |Activo         |
|2          |Luis  |IT          |Activo         |
|4          |María |RRHH        |Activo         |
|5          |Sofía |Marketing   |Activo         |
|6          |José  |IT          |Activo         |
|8          |Daniel|Finanzas    |Activo         |
|9          |Paola |RRHH        |Activo         |
+-----------+------+------------+---------------+



In [16]:
#TRANSFORMACION 2 - WITHCOLUMN: COLUMNA DERIVADA
#BONO ESTIMADO = SALARIO * (DESEMPEÑO / 100) * 0.10

df_enriquecido = df_activos.withColumn(
    "bono_estimado",
    F.round(F.col("salario") * (F.col("desempeño") / 100) * 0.10, 2)
)

print("Columna 'bono_estimado' creada---------------------")
df_enriquecido.select("id_empleado", "nombre", "salario", "desempeño", "bono_estimado").show(truncate=False)

Columna 'bono_estimado' creada---------------------
+-----------+------+-------+---------+-------------+
|id_empleado|nombre|salario|desempeño|bono_estimado|
+-----------+------+-------+---------+-------------+
|1          |Ana   |550.0  |85       |46.75        |
|2          |Luis  |900.0  |92       |82.8         |
|4          |María |750.0  |88       |66.0         |
|5          |Sofía |700.0  |78       |54.6         |
|6          |José  |950.0  |95       |90.25        |
|8          |Daniel|800.0  |82       |65.6         |
|9          |Paola |720.0  |90       |64.8         |
+-----------+------+-------+---------+-------------+



In [17]:
#TRANSFORMACION 3 - GROUPBY + ORDERBY: RESUMEN POR DEPARTAMENTO

resumen_departamento = df_activos.groupBy("departamento").agg(
    F.round(F.avg("salario"), 2).alias("salario_promedio"),
    F.round(F.avg("desempeño"), 2).alias("desempeño_promedio"),
    F.round(F.avg("satisfaccion"), 2).alias("satisfaccion_promedio"),
    F.count("*").alias("cantidad_empleados")
).orderBy(F.desc("salario_promedio"))

print("Resumen por departamento----------------")
resumen_departamento.show(truncate=False)

Resumen por departamento----------------
+------------+----------------+------------------+---------------------+------------------+
|departamento|salario_promedio|desempeño_promedio|satisfaccion_promedio|cantidad_empleados|
+------------+----------------+------------------+---------------------+------------------+
|IT          |925.0           |93.5              |5.0                  |2                 |
|Finanzas    |800.0           |82.0              |4.0                  |1                 |
|RRHH        |735.0           |89.0              |4.5                  |2                 |
|Marketing   |700.0           |78.0              |3.0                  |1                 |
|Ventas      |550.0           |85.0              |4.0                  |1                 |
+------------+----------------+------------------+---------------------+------------------+



In [18]:
#TRANSFORMACION 4 - GROUPBY + ORDERBY: RESUMEN POR CATEGORIA DE ANTIGÜEDAD

resumen_antiguedad = df_activos.groupBy("categoria_antiguedad").agg(
    F.count("*").alias("cantidad_empleados"),
    F.round(F.avg("horas_extra"), 2).alias("horas_extra_promedio")
).orderBy(F.desc("cantidad_empleados"))

print("Resumen por categoria de antiguedad----------------")
resumen_antiguedad.show(truncate=False)

Resumen por categoria de antiguedad----------------
+--------------------+------------------+--------------------+
|categoria_antiguedad|cantidad_empleados|horas_extra_promedio|
+--------------------+------------------+--------------------+
|Junior              |4                 |4.5                 |
|Senior              |3                 |4.0                 |
+--------------------+------------------+--------------------+



In [19]:
#TRANSFORMACION 5 - JOIN CON TABLA DE DIMENSION (NIVEL DE PRIORIDAD POR DEPARTAMENTO)

departamentos_prioridad = [
    ("Ventas", "Comercial", "Alto"),
    ("IT", "Tecnologia", "Alto"),
    ("Finanzas", "Administrativo", "Medio"),
    ("RRHH", "Administrativo", "Medio"),
    ("Marketing", "Comercial", "Alto"),
]

df_prioridad = spark.createDataFrame(
    departamentos_prioridad, ["departamento", "area", "nivel_prioridad"]
)

print("Tabla de prioridad por departamento: -----------------")
df_prioridad.show()

df_con_prioridad = df_enriquecido.join(df_prioridad, on="departamento", how="left")
print(f"Filas antes: {df_enriquecido.count()}")
print(f"Filas despues: {df_con_prioridad.count()}")

print("Datos con area y nivel de prioridad: ----------")
df_con_prioridad.select(
    "id_empleado", "nombre", "departamento", "area", "nivel_prioridad", "bono_estimado"
).orderBy(F.desc("bono_estimado")).show(10, truncate=False)

Tabla de prioridad por departamento: -----------------
+------------+--------------+---------------+
|departamento|          area|nivel_prioridad|
+------------+--------------+---------------+
|      Ventas|     Comercial|           Alto|
|          IT|    Tecnologia|           Alto|
|    Finanzas|Administrativo|          Medio|
|        RRHH|Administrativo|          Medio|
|   Marketing|     Comercial|           Alto|
+------------+--------------+---------------+

Filas antes: 7
Filas despues: 7
Datos con area y nivel de prioridad: ----------
+-----------+------+------------+--------------+---------------+-------------+
|id_empleado|nombre|departamento|area          |nivel_prioridad|bono_estimado|
+-----------+------+------------+--------------+---------------+-------------+
|6          |José  |IT          |Tecnologia    |Alto           |90.25        |
|2          |Luis  |IT          |Tecnologia    |Alto           |82.8         |
|4          |María |RRHH        |Administrativo|Medio  

In [20]:
#CREAR VISTA TEMPORAL
df_con_prioridad.createOrReplaceTempView("empleados_analiticos")
print("Vista temporal 'empleados_analiticos' creada")

Vista temporal 'empleados_analiticos' creada


In [21]:
#CONSULTA SQL - RESUMEN POR DEPARTAMENTO Y NIVEL DE PRIORIDAD

consulta = """
    SELECT
        departamento,
        area,
        nivel_prioridad,
        COUNT(*) AS cantidad_empleados,
        ROUND(AVG(salario), 2) AS salario_promedio,
        ROUND(AVG(`desempeño`), 2) AS desempeno_promedio,
        ROUND(SUM(bono_estimado), 2) AS bono_total
    FROM empleados_analiticos
    GROUP BY departamento, area, nivel_prioridad
    ORDER BY salario_promedio DESC
"""

resultado_sql = spark.sql(consulta)
print("CONSULTA SQL - RESUMEN POR DEPARTAMENTO -----------------")
resultado_sql.show(truncate=False)

CONSULTA SQL - RESUMEN POR DEPARTAMENTO -----------------
+------------+--------------+---------------+------------------+----------------+------------------+----------+
|departamento|area          |nivel_prioridad|cantidad_empleados|salario_promedio|desempeno_promedio|bono_total|
+------------+--------------+---------------+------------------+----------------+------------------+----------+
|IT          |Tecnologia    |Alto           |2                 |925.0           |93.5              |173.05    |
|Finanzas    |Administrativo|Medio          |1                 |800.0           |82.0              |65.6      |
|RRHH        |Administrativo|Medio          |2                 |735.0           |89.0              |130.8     |
|Marketing   |Comercial     |Alto           |1                 |700.0           |78.0              |54.6      |
|Ventas      |Comercial     |Alto           |1                 |550.0           |85.0              |46.75     |
+------------+--------------+---------------+-

In [22]:
#SEGUNDA CONSULTA SQL - EMPLEADOS CON MEJOR DESEMPEÑO

consulta_top = """
    SELECT nombre, departamento, `desempeño`, satisfaccion, bono_estimado
    FROM empleados_analiticos
    ORDER BY `desempeño` DESC
    LIMIT 5
"""

top_empleados = spark.sql(consulta_top)
print("CONSULTA SQL - TOP 5 EMPLEADOS POR DESEMPEÑO -----------------")
top_empleados.show(truncate=False)

CONSULTA SQL - TOP 5 EMPLEADOS POR DESEMPEÑO -----------------
+------+------------+---------+------------+-------------+
|nombre|departamento|desempeño|satisfaccion|bono_estimado|
+------+------------+---------+------------+-------------+
|José  |IT          |95       |5           |90.25        |
|Luis  |IT          |92       |5           |82.8         |
|Paola |RRHH        |90       |5           |64.8         |
|María |RRHH        |88       |4           |66.0         |
|Ana   |Ventas      |85       |4           |46.75        |
+------+------------+---------+------------+-------------+



In [23]:
#VER PLAN DE EJECUCION

print("Plan de ejecucion formateado:----------------------")
resultado_sql.explain("formatted")

Plan de ejecucion formateado:----------------------
== Physical Plan ==
AdaptiveSparkPlan (16)
+- Sort (15)
   +- Exchange (14)
      +- HashAggregate (13)
         +- HashAggregate (12)
            +- Project (11)
               +- SortMergeJoin LeftOuter (10)
                  :- Sort (5)
                  :  +- Exchange (4)
                  :     +- Project (3)
                  :        +- Filter (2)
                  :           +- Scan csv  (1)
                  +- Sort (9)
                     +- Exchange (8)
                        +- Filter (7)
                           +- Scan ExistingRDD (6)


(1) Scan csv 
Output [4]: [departamento#174, desempeño#176, salario#177, estado_empleado#179]
Batched: false
Location: InMemoryFileIndex [file:/content/datos/procesados/dataset_limpio_HRTech_People_Analytics.csv]
PushedFilters: [IsNotNull(estado_empleado), EqualTo(estado_empleado,Activo)]
ReadSchema: struct<departamento:string,desempeño:int,salario:double,estado_empleado:string>

(2)

In [24]:
#INSPECCIONAR PARTICIONES ACTUALES
print(f"Particiones actuales: {df_con_prioridad.rdd.getNumPartitions()}")

Particiones actuales: 1


In [25]:
#REPARTICIONAR EN 4

df_particionado = df_con_prioridad.repartition(4)
print(f"Nuevas particiones: {df_particionado.rdd.getNumPartitions()}")

Nuevas particiones: 4


In [26]:
#GUARDAR EN PARQUET

ruta_parquet = DATA_ANALYTICS / "empleados_analiticos_parquet"

df_particionado.write \
    .mode("overwrite") \
    .parquet(str(ruta_parquet))

print(f"Parquet guardado en: {ruta_parquet}")

Parquet guardado en: /content/datos/analiticos/empleados_analiticos_parquet


In [27]:
#LEER PARQUET PARA VERIFICAR
df_leido = spark.read.parquet(str(ruta_parquet))

print(f"Filas leidas: {df_leido.count()}")
print("Primeras filas: -----------------")
df_leido.show(5, truncate=False)

Filas leidas: 7
Primeras filas: -----------------
+------------+-----------+------+----+---------+-------+--------------------+---------------+-------------+---------------+----------------------+-----+-----------+------------------+------------+-----------+----------------+-----------------+----------+--------------------+--------------+---------------------+-------------+--------------+---------------+
|departamento|id_empleado|nombre|edad|desempeño|salario|tiempo_empresa_anios|estado_empleado|salario_anual|id_capacitacion|curso                 |horas|resultado  |horas_capacitacion|satisfaccion|horas_extra|salario_por_hora|anios_redondeados|es_antiguo|categoria_antiguedad|rango_edad    |requiere_capacitacion|bono_estimado|area          |nivel_prioridad|
+------------+-----------+------+----+---------+-------+--------------------+---------------+-------------+---------------+----------------------+-----+-----------+------------------+------------+-----------+----------------+---------

In [28]:
#AGREGACION CON PANDAS (CON MEDICION DE TIEMPO)

inicio_pd = time.perf_counter()

df_pd = pd.read_csv(DATA_PROCESSED / "dataset_limpio_HRTech_People_Analytics.csv")
pd_departamento = (
    df_pd[df_pd["estado_empleado"] == "Activo"]
    .groupby("departamento")["salario"]
    .mean()
    .round(2)
    .reset_index()
)
pd_departamento.columns = ["departamento", "salario_promedio_pandas"]

fin_pd = time.perf_counter()
tiempo_pandas = fin_pd - inicio_pd

print(f"Tiempo Pandas: {tiempo_pandas:.4f} segundos")
print("Pandas - salario promedio por departamento")
print(pd_departamento)

Tiempo Pandas: 0.1093 segundos
Pandas - salario promedio por departamento
  departamento  salario_promedio_pandas
0     Finanzas                    800.0
1           IT                    925.0
2    Marketing                    700.0
3         RRHH                    735.0
4       Ventas                    550.0


In [29]:
#MISMA AGREGACION CON SPARK (CON MEDICION DE TIEMPO)

inicio_spark = time.perf_counter()

sp_departamento = (
    df_activos.groupBy("departamento")
    .agg(F.round(F.avg("salario"), 2).alias("salario_promedio_spark"))
    .orderBy("departamento")
    .toPandas()
)

fin_spark = time.perf_counter()
tiempo_spark = fin_spark - inicio_spark

print(f"Tiempo Spark: {tiempo_spark:.4f} segundos")
print("\nSpark - salario promedio por departamento")
print(sp_departamento)

Tiempo Spark: 1.4692 segundos

Spark - salario promedio por departamento
  departamento  salario_promedio_spark
0     Finanzas                   800.0
1           IT                   925.0
2    Marketing                   700.0
3         RRHH                   735.0
4       Ventas                   550.0


In [30]:
#COMPARAR RESULTADOS Y TIEMPOS

print("Comparacion Pandas vs Spark (resultados)")
comparacion = pd.merge(pd_departamento, sp_departamento, on="departamento")
print(comparacion)

print("\nComparacion de tiempos de ejecucion")
print(f"Pandas: {tiempo_pandas:.4f} s")
print(f"Spark:  {tiempo_spark:.4f} s")

if tiempo_pandas < tiempo_spark:
    diferencia = tiempo_spark / tiempo_pandas
    print(f"\nPandas fue {diferencia:.1f}x mas rapido que Spark en esta prueba.")
else:
    diferencia = tiempo_pandas / tiempo_spark
    print(f"\nSpark fue {diferencia:.1f}x mas rapido que Pandas en esta prueba.")

Comparacion Pandas vs Spark (resultados)
  departamento  salario_promedio_pandas  salario_promedio_spark
0     Finanzas                    800.0                   800.0
1           IT                    925.0                   925.0
2    Marketing                    700.0                   700.0
3         RRHH                    735.0                   735.0
4       Ventas                    550.0                   550.0

Comparacion de tiempos de ejecucion
Pandas: 0.1093 s
Spark:  1.4692 s

Pandas fue 13.4x mas rapido que Spark en esta prueba.


In [31]:
#REPORTE DE ARQUITECTURA - GUIA 3 - PROYECTO HRTECH

print("\nREPORTE DE ARQUITECTURA - GUIA 3 - HRTECH PEOPLE ANALYTICS")
print("=" * 60)
print(f"Version de Spark: {spark.version}")
print(f"Dataset de entrada: dataset_limpio_HRTech_People_Analytics.csv")
print(f"Filas procesadas: {df_activos.count()}")
print(f"Columnas originales: {len(df.columns)}")
print(f"Columnas finales: {len(df_particionado.columns)}")
print(f"Particiones iniciales: {df.rdd.getNumPartitions()}")
print(f"Particiones finales: {df_particionado.rdd.getNumPartitions()}")
print(f"Salida Parquet: {ruta_parquet}")
print(f"Formato: Parquet")
print(f"Tiempo Pandas (agregacion): {tiempo_pandas:.4f} s")
print(f"Tiempo Spark (agregacion): {tiempo_spark:.4f} s")
print("=" * 60)


REPORTE DE ARQUITECTURA - GUIA 3 - HRTECH PEOPLE ANALYTICS
Version de Spark: 4.0.5
Dataset de entrada: dataset_limpio_HRTech_People_Analytics.csv
Filas procesadas: 7
Columnas originales: 22
Columnas finales: 25
Particiones iniciales: 1
Particiones finales: 4
Salida Parquet: /content/datos/analiticos/empleados_analiticos_parquet
Formato: Parquet
Tiempo Pandas (agregacion): 0.1093 s
Tiempo Spark (agregacion): 1.4692 s


In [32]:
# DESCARGAR PARQUET (ZIP automático)

!zip -r /content/empleados_analiticos_parquet.zip /content/  Datos/analiticos/empleados_analiticos_parquet/


from google.colab import files

files.download("/content/empleados_analiticos_parquet.zip")

print("Parquet descargado!")

	zip warning: name not matched: Datos/analiticos/empleados_analiticos_parquet/
  adding: content/ (stored 0%)
  adding: content/.config/ (stored 0%)
  adding: content/.config/active_config (stored 0%)
  adding: content/.config/.last_survey_prompt.yaml (stored 0%)
  adding: content/.config/config_sentinel (stored 0%)
  adding: content/.config/.last_opt_in_prompt.yaml (stored 0%)
  adding: content/.config/.last_update_check.json (deflated 22%)
  adding: content/.config/configurations/ (stored 0%)
  adding: content/.config/configurations/config_default (deflated 15%)
  adding: content/.config/logs/ (stored 0%)
  adding: content/.config/logs/2026.10.26/ (stored 0%)
  adding: content/.config/logs/2026.10.26/22.18.45.880512.log (deflated 58%)
  adding: content/.config/logs/2026.10.26/22.18.15.518759.log (deflated 92%)
  adding: content/.config/logs/2026.10.26/22.18.38.571729.log (deflated 58%)
  adding: content/.config/logs/2026.10.26/22.18.52.867204.log (deflated 57%)
  adding: content/.con

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Parquet descargado!
